# CIC-IDS-2018 — NNIDS Pipeline

Minimal notebook to reproduce the full dataset pipeline on Colab:
1. **Setup** — system packages, Java 8, CICFlowMeter, AWS CLI, jnetpcap
2. **Clone** the pipeline repo
3. **Configure** target days and run parameters
4. **Run** `main.py`


In [ ]:
%%bash
set -euo pipefail
apt-get update -qq
apt-get install -y openjdk-8-jdk unzip zip unrar git gradle maven \
                   build-essential patchelf libpcap0.8 libpcap0.8-dev


In [ ]:
%%bash
set -euo pipefail
curl -sSL "https://awscli.amazonaws.com/awscli-exe-linux-x86_64-2.0.30.zip" -o awscliv2.zip
unzip -o awscliv2.zip
sudo ./aws/install
rm -rf aws awscliv2.zip
aws --version


In [ ]:
%%bash
set -euo pipefail
REPO_DIR=/content/nnids_pipeline
if [ ! -d "$REPO_DIR/.git" ]; then
  git clone https://github.com/devedale/cic-ids-2018-labeling.git "$REPO_DIR"
else
  echo 'Repo already cloned, pulling latest...'
  git -C "$REPO_DIR" pull --ff-only
fi
cd "$REPO_DIR/nnids_pipeline"
pip install -q -r requirements.txt
echo 'Done.'


In [ ]:
%%bash
set -euo pipefail
REPO_DIR=/content/CICFlowMeter
rm -rf "$REPO_DIR"
git clone https://github.com/UNBCIC/CICFlowMeter "$REPO_DIR"
cd "$REPO_DIR"
[ ! -f gradlew ] && gradle wrapper --gradle-version 7.6
chmod +x gradlew

# copy native jnetpcap libs
mkdir -p /content/native_libs
python3 - <<'PY'
from pathlib import Path; import shutil
jnet = Path('/content/CICFlowMeter/jnetpcap/linux/jnetpcap-1.4.r1425')
nd   = Path('/content/native_libs')
libs = sorted(jnet.rglob('*.so'))
if not libs: raise SystemExit(f'No .so files in {jnet}')
for lib in libs:
    lib.chmod(0o755); shutil.copy2(lib, nd / lib.name)
for cand in [Path('/usr/lib/x86_64-linux-gnu/libpcap.so.0.8'),
             Path('/usr/lib/x86_64-linux-gnu/libpcap.so.1'),
             Path('/usr/lib/x86_64-linux-gnu/libpcap.so')]:
    if cand.exists():
        compat = nd / 'libpcap.so.0.8'
        if not compat.exists(): compat.symlink_to(cand)
        break
print('native libs:', sorted(p.name for p in nd.iterdir()))
PY


In [ ]:
%%bash
set -euo pipefail
mvn install:install-file \
  -Dfile=/content/CICFlowMeter/jnetpcap/linux/jnetpcap-1.4.r1425/jnetpcap.jar \
  -DgroupId=org.jnetpcap -DartifactId=jnetpcap -Dversion=1.4.1 \
  -Dpackaging=jar -DgeneratePom=true -q
echo 'jnetpcap installed in local Maven repo'


In [ ]:
!java -version
!aws --version
!/content/CICFlowMeter/gradlew -v


## Configuration

Edit the cells below to select which days to process and tune run parameters.

Available days and their attack labels:
| Day | Attacks |
|-----|---------|
| Wednesday-14-02-2018 | FTP-BruteForce, SSH-BruteForce |
| Thursday-15-02-2018  | DoS-GoldenEye, DoS-Slowloris |
| Friday-16-02-2018    | DoS-SlowHTTPTest, DoS-Hulk |
| Tuesday-20-02-2018   | DDoS-LOIC-HTTP, DDoS-LOIC-UDP |
| Wednesday-21-02-2018 | DDoS-LOIC-UDP, DDoS-HOIC |
| Thursday-22-02-2018  | BruteForce-Web, BruteForce-XSS, SQL-Injection |
| Friday-23-02-2018    | BruteForce-Web, BruteForce-XSS, SQL-Injection |
| Wednesday-28-02-2018 | Infiltration |
| Thursday-01-03-2018  | Infiltration |
| Friday-02-03-2018    | Bot |


In [ ]:
# ── Days to process ───────────────────────────────────────────────────────
DAYS = [
    'Thursday-22-02-2018',   # BruteForce-Web, BruteForce-XSS, SQL-Injection
    'Friday-23-02-2018',     # BruteForce-Web, BruteForce-XSS, SQL-Injection
    'Friday-02-03-2018',     # Bot
]

# ── Run parameters ─────────────────────────────────────────────────────────
FORCE_RERUN   = False    # True -> re-download and reprocess even if cached
SAMPLE_SIZE   = 20_000   # max rows after preprocessing; None = keep all
CACHE_ENABLED = True     # persist preprocessed.csv in preprocessed_cache/

print('Days    :', DAYS)
print('Force   :', FORCE_RERUN)
print('Sample  :', SAMPLE_SIZE)
print('Cache   :', CACHE_ENABLED)


## Run pipeline

Equivalent to:
```bash
python main.py --days <DAYS> [--force] [--sample N] [--no-cache]
```


In [ ]:
import sys
from pathlib import Path

PIPELINE_ROOT = Path('/content/nnids_pipeline/nnids_pipeline')
if str(PIPELINE_ROOT) not in sys.path:
    sys.path.insert(0, str(PIPELINE_ROOT))

# Override settings before importing
import importlib, configs.settings as _s
_s.DAYS          = DAYS
_s.SAMPLE_SIZE   = SAMPLE_SIZE
_s.CACHE_ENABLED = CACHE_ENABLED

from core.ingestion import Ingestion
from core.preprocessing import preprocess, clean_temp

ing    = Ingestion(base_dir=PIPELINE_ROOT)
raw    = ing.run(days=DAYS, force_rerun=FORCE_RERUN)
raw_df = raw['dataframe']

preproc_df = preprocess(
    raw_df,
    sample_size=SAMPLE_SIZE,
    cache=CACHE_ENABLED,
    cache_dir=_s.CACHE_DIR,
)

clean_temp(PIPELINE_ROOT)

print('Pipeline terminata')
print('Record totali       :', len(raw_df))
print('Record preprocessati:', len(preproc_df))
print('Cache in            :', _s.CACHE_DIR)


## Results


In [ ]:
import pandas as pd

print('=== Label distribution (raw) ===')
print(raw_df['Label'].value_counts().to_string())

if '_source_day' in raw_df.columns:
    print('\n=== Per-day breakdown ===')
    display(
        raw_df.groupby(['_source_day', 'Label'])
        .size().reset_index(name='count')
        .sort_values(['_source_day', 'count'], ascending=[True, False])
    )

print('\n=== Preprocessed shape:', preproc_df.shape, '===')
display(preproc_df.head())
